# H1 · Coulomb repulsion from a *local* equation: a first tutorial
**Series H, Post 1** &nbsp;|&nbsp; tutorial notebook for the post of the same name

**The problem.** Electrons repel each other with the Coulomb force $1/r$. Summing that force over every *pair* of $P$ charges costs $P^2$ operations and every charge "talks to" every other one. **The idea tested here:** replace the pair sum by a *helper field* $\phi$ on a grid that obeys a **local** equation (Poisson's equation, $-\nabla^2\phi=4\pi\rho$): each grid point only needs its six neighbours. In three dimensions the solution of that local equation *is* the Coulomb potential, so nothing is lost in the continuum. On a grid the potential differs a little, and the notebook measures how much, using **harmonium** (two electrons in a harmonic trap, whose exact energy is $2$ hartree) as the test.

> **You need** high-school calculus and the idea of a force between charges. Every term (potential, Laplacian, stencil, Green's function, boundary condition, Hartree–Fock, convergence order) is explained when it first appears. The post has a glossary.
>
> **Two parts.** **Part I** needs NumPy, SciPy and matplotlib and takes about two minutes on a CPU. **Part II** is optional: independent checks with SciPy only.
>
> **How to use it.** Each section starts with a short **Stop and predict** question. Answer it on paper before you run the cell.

| Section | What you do |
|---|---|
| 1 | the cost of the pair sum |
| 2 | the local equation and its stencil, solved for one charge |
| 3 | the lattice Green function: what the grid does to $1/r$ |
| 4 | boundaries: zero walls versus open boundary values |
| 5 | harmonium: the exact and mean-field energies |
| 6 | mean-field harmonium on a grid; error budget and cancellation |
| 7 | the exact two-electron problem with the grid interaction |
| 8 | the self-term |
| 9 | the cost of the local solve |

## Helper functions
One cell with the numerics (lattice Green function, the Poisson solver, the harmonium references, the grid Hartree–Fock) and one with the figure code. You do not need to read them to follow the text; the sections below call them one at a time.

In [ ]:
"""Numerics for the 'Coulomb from a local constraint' tutorial: lattice Green function, grid Poisson solver, harmonium references."""
import time, math, numpy as np, scipy.sparse as sp, scipy.sparse.linalg as spla
from scipy.special import ive, erf
from scipy.fft import dstn, idstn
from scipy.integrate import cumulative_trapezoid
from scipy.linalg import eigh_tridiagonal
trapz = getattr(np, "trapezoid", None) or np.trapz
G0 = 0.2527309                      # lattice Green function at the origin (simple cubic): G(0,0,0) = 0.2527310...
def lattice_G(ns, nt=3000, tmin=1e-7, tmax=1e8):
    """infinite simple-cubic lattice Green function  G(n) = int_0^inf exp(-6t) I_n1(2t) I_n2(2t) I_n3(2t) dt  (+ analytic tail)"""
    ns = np.abs(np.asarray(ns, int)); M = len(ns); s = np.linspace(np.log(tmin), np.log(tmax), nt); t = np.exp(s); ds = s[1] - s[0]; out = np.zeros(M); w = np.full(nt, ds); w[0] = w[-1] = ds / 2
    for k in range(0, M, 2000):
        blk = ns[k:k + 2000]; f = np.ones((len(blk), nt))
        for d in range(3): f *= ive(blk[:, d][:, None], 2 * t[None, :])
        out[k:k + 2000] = (f * t[None, :] * w[None, :]).sum(axis=1)
    return out + 2 / (4 * np.pi) ** 1.5 / np.sqrt(tmax)
def poisson_dirichlet(rho, h, bnd=None):
    """solve (6 phi - sum of 6 neighbours)/h^2 = 4 pi rho on the N^3 interior; bnd(X,Y,Z) gives the potential on the walls (None = zero walls); a fast sine transform inverts the same local operator"""
    N = rho.shape[0]; rhs = 4 * np.pi * rho.copy()
    if bnd is not None:
        c = (np.arange(N) - (N - 1) / 2) * h; xb = (N + 1) / 2 * h; A, B = np.meshgrid(c, c, indexing='ij'); W = np.full_like(A, xb)
        for sgn, sl in ((-1, 0), (1, -1)): rhs[sl, :, :] += bnd(sgn * W, A, B) / h ** 2; rhs[:, sl, :] += bnd(A, sgn * W, B) / h ** 2; rhs[:, :, sl] += bnd(A, B, sgn * W) / h ** 2
    k = np.arange(1, N + 1); lam1 = (2 - 2 * np.cos(np.pi * k / (N + 1))) / h ** 2; lam = lam1[:, None, None] + lam1[None, :, None] + lam1[None, None, :]
    return idstn(dstn(rhs, type=1) / lam, type=1)
def multipole_bc(Q, p):
    def f(X, Y, Z): r = np.sqrt(X ** 2 + Y ** 2 + Z ** 2); return Q / r + (p[0] * X + p[1] * Y + p[2] * Z) / r ** 3
    return f
def cg_solve(A, b, tol):
    its = [0]
    def cb(xk): its[0] += 1
    try: x, info = spla.cg(A, b, rtol=tol, callback=cb, maxiter=5000)
    except TypeError: x, info = spla.cg(A, b, tol=tol, callback=cb, maxiter=5000)
    return x, its[0]
def stencil_matrix(N, h):
    I = sp.identity(N, format='csr'); T1 = (sp.diags([2 * np.ones(N), -np.ones(N - 1), -np.ones(N - 1)], [0, 1, -1]) / h ** 2).tocsr()
    return (sp.kron(sp.kron(T1, I), I) + sp.kron(sp.kron(I, T1), I) + sp.kron(sp.kron(I, I), T1)).tocsr()
def radial_reference(w=0.5, R=14.0, n=12000, parts=False):
    """exact two-electron harmonium energy (relative coordinate) and spherical restricted Hartree-Fock energy"""
    r = np.linspace(0, R, n + 2)[1:-1]; dr = r[1] - r[0]
    def lowest(pot, coef):
        d = 2 * coef / dr ** 2 + pot; e = -coef / dr ** 2 * np.ones(n - 1); ev, V = eigh_tridiagonal(d, e, select='i', select_range=(0, 0)); return ev[0], V[:, 0]
    eps, _ = lowest(w ** 2 * r ** 2 / 4 + 1 / r, 1.0); Eex = 1.5 * w + eps
    u = np.exp(-w * r ** 2 / 2) * r; u /= np.sqrt(np.sum(u ** 2) * dr)
    def vh(u):
        nrho = 2 * u ** 2 / (4 * np.pi * r ** 2); q = cumulative_trapezoid(nrho * 4 * np.pi * r ** 2, r, initial=0); tot = trapz(nrho * 4 * np.pi * r, r); out = tot - cumulative_trapezoid(nrho * 4 * np.pi * r, r, initial=0); return nrho, q / r + out
    for it in range(300):
        nrho, v = vh(u); e, un = lowest(0.5 * w ** 2 * r ** 2 + 0.5 * v, 0.5); un = np.abs(un) / np.sqrt(np.sum(un ** 2) * dr)
        if np.abs(un - u).max() < 1e-13: u = un; break
        u = 0.5 * u + 0.5 * un
    nrho, v = vh(u); d2u = (np.r_[u[1:], 0] - 2 * u + np.r_[0, u[:-1]]) / dr ** 2; kin = np.sum(u * (-0.5 * d2u)) * dr; pot = np.sum(u ** 2 * 0.5 * w ** 2 * r ** 2) * dr; Jee = 0.25 * trapz(nrho * v * 4 * np.pi * r ** 2, r)
    if parts: return Eex, 2 * (kin + pot) + Jee, 2 * (kin + pot), Jee
    return Eex, 2 * (kin + pot) + Jee
def cube_avg_inv_r(n=400):
    def mid(n):
        c = (np.arange(n) + 0.5) / n - 0.5; tot = 0.0
        for i in range(n): tot += np.sum(1 / np.sqrt(c[i] ** 2 + c[:, None] ** 2 + c[None, :] ** 2))
        return tot / n ** 3
    return 2 * mid(n) - mid(n // 2)
CELL0 = 2.380079; SC = 4 * np.pi * G0 - CELL0
def grid_hf(h, L, bc='multipole', w=0.5, tol=1e-9, maxit=60, selfcorr=False, parts=False):
    """self-consistent Hartree-Fock for harmonium on an N^3 grid; the Hartree potential comes from the local constraint"""
    N = int(round(2 * L / h)) + 1; c = (np.arange(N) - (N - 1) / 2) * h; X, Y, Z = np.meshgrid(c, c, c, indexing='ij'); r2 = X ** 2 + Y ** 2 + Z ** 2
    I = sp.identity(N, format='csr'); T1 = (sp.diags([-2 * np.ones(N), np.ones(N - 1), np.ones(N - 1)], [0, 1, -1]) / h ** 2).tocsr(); Lap = sp.kron(sp.kron(T1, I), I) + sp.kron(sp.kron(I, T1), I) + sp.kron(sp.kron(I, I), T1)
    Tk = -0.5 * Lap.tocsr(); Vtrap = (0.5 * w ** 2 * r2).ravel(); phi = np.exp(-w * r2 / 2).ravel(); phi /= np.sqrt(np.sum(phi ** 2) * h ** 3); Eold = 0
    def hartree(phi):
        n = 2 * (phi ** 2).reshape(N, N, N); bnd = None if bc == 'zero' else multipole_bc(np.sum(n) * h ** 3, [0, 0, 0]); vH = poisson_dirichlet(n, h, bnd)
        if selfcorr: vH = vH - SC * h ** 2 * n
        return n, vH
    for it in range(maxit):
        n, vH = hartree(phi); e, V = spla.eigsh(Tk + sp.diags(Vtrap + 0.5 * vH.ravel()), k=1, which='SA', v0=phi, tol=1e-10)
        new = V[:, 0] * np.sign(V[:, 0].sum()); new /= np.sqrt(np.sum(new ** 2) * h ** 3); phi = 0.5 * phi + 0.5 * new if it > 0 else new; phi /= np.sqrt(np.sum(phi ** 2) * h ** 3)
        n, vH = hartree(phi); kin = np.sum(phi * (Tk @ phi)) * h ** 3; pot = np.sum(phi ** 2 * Vtrap) * h ** 3; Jee = 0.25 * np.sum(n * vH) * h ** 3; E = 2 * (kin + pot) + Jee
        if abs(E - Eold) < tol: break
        Eold = E
    if parts: return E, it + 1, N, 2 * (kin + pot), Jee
    return E, it + 1, N
def relative_problem(h, Lr=10.0, w=0.5, potential='lattice'):
    N = int(round(2 * Lr / h)) + 1; m = (N - 1) // 2; n1 = np.arange(N) - m; I = sp.identity(N, format='csr'); T1 = (sp.diags([-2 * np.ones(N), np.ones(N - 1), np.ones(N - 1)], [0, 1, -1]) / h ** 2).tocsr()
    Lap = sp.kron(sp.kron(T1, I), I) + sp.kron(sp.kron(I, T1), I) + sp.kron(sp.kron(I, I), T1); X, Y, Z = np.meshgrid(n1, n1, n1, indexing='ij'); nn = np.stack([np.abs(X).ravel(), np.abs(Y).ravel(), np.abs(Z).ravel()], axis=1); r2 = (nn ** 2).sum(axis=1) * h ** 2
    if potential == 'lattice': uk, inv = np.unique(np.sort(nn, axis=1), axis=0, return_inverse=True); V = (4 * np.pi * lattice_G(uk) / h)[inv.ravel()]
    else: rr = np.sqrt(r2); V = np.where(rr > 0, 1 / np.where(rr > 0, rr, 1), 0.0); V[r2 == 0] = CELL0 / h
    H = -Lap + sp.diags(0.25 * w ** 2 * r2 + V); return 1.5 * w + spla.eigsh(H.tocsc(), k=1, which='SA', tol=1e-10)[0][0], N

In [ ]:
import math, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
C = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
def fig_stencil():
    fig, ax = plt.subplots(1, 3, figsize=(11.3, 3.4), gridspec_kw=dict(width_ratios=[1.15, 1, 1.1]))
    N = 41; h = 1.0; x = np.arange(N); L = h * (N - 1); rho = np.zeros(N); rho[N // 2] = 1.0
    A = (np.diag(2 * np.ones(N)) - np.diag(np.ones(N - 1), 1) - np.diag(np.ones(N - 1), -1)); phi = np.linalg.solve(A, 4 * np.pi * rho * 0.05)
    ax[0].plot(x, phi, "o-", color=C["blue"], ms=3); ax[0].bar([N // 2], [phi.max() * 0.35], width=0.8, color=C["red"], alpha=0.7); ax[0].text(N // 2 + 1, phi.max() * 0.4, "charge", color=C["red"], fontsize=8)
    ax[0].set(xlabel="grid point", ylabel="potential", title="(a) 1D: one charge, the local equation\n(each point tied only to its neighbours)"); ax[0].set_yticks([])
    pts = np.array([[0, 0], [1, 0], [-1, 0], [0, 1], [0, -1]]); ax[1].scatter(pts[1:, 0], pts[1:, 1], s=300, color=C["blue"], zorder=3); ax[1].scatter([0], [0], s=420, color=C["orange"], zorder=3)
    for p in pts[1:]: ax[1].plot([0, p[0]], [0, p[1]], color="k", lw=1.2, zorder=2)
    ax[1].text(0, 0, r"$\phi_0$", ha="center", va="center", fontsize=9); ax[1].set(xlim=(-1.8, 1.8), ylim=(-1.6, 1.8), aspect="equal", title="(b) 2D picture: the 5-point stencil\n(3D uses 6 neighbours)"); ax[1].axis("off")
    ax[1].text(0, -1.45, r"$4\phi_0-\sum_{\rm nb}\phi=4\pi h^2\rho_0$", ha="center", fontsize=9)
    X = np.arange(21); Y = np.arange(21); XX, YY = np.meshgrid(X, Y); r = np.sqrt((XX - 10) ** 2 + (YY - 10) ** 2 + 0.25); im = ax[2].imshow(1 / r, cmap="viridis", origin="lower"); ax[2].set(title="(c) the potential of one charge fills\nthe whole grid (2D slice of 3D, $1/r$)", xticks=[], yticks=[]); plt.colorbar(im, ax=ax[2], fraction=0.045); plt.tight_layout(); return fig
def fig_lattice(lattice_G):
    ns = np.array([2, 3, 4, 6, 8, 12, 16, 24, 32, 48]); ga = lattice_G(np.stack([ns, 0 * ns, 0 * ns], 1)); gd = lattice_G(np.stack([ns, ns, ns], 1))
    ra, rd = ns, ns * np.sqrt(3); ea, ed = 4 * np.pi * ra * ga - 1, 4 * np.pi * rd * gd - 1
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7)); ax[0].plot(ra, 4 * np.pi * ra * ga, "o-", color=C["blue"], label="along an axis"); ax[0].plot(rd, 4 * np.pi * rd * gd, "s-", color=C["orange"], label="along the body diagonal"); ax[0].axhline(1, color="k", lw=0.8)
    ax[0].set(xscale="log", xlabel="distance in grid units", ylabel=r"$4\pi\,r\,G(r)$  (1 = exact $1/r$)", title="(a) the lattice potential vs. the exact $1/r$"); ax[0].legend(frameon=False, fontsize=8)
    nn = np.logspace(0.3, 1.7, 30); ax[1].loglog(ra, np.abs(ea), "o", color=C["blue"], label="axis"); ax[1].loglog(rd, np.abs(ed), "s", color=C["orange"], label="body diagonal"); ax[1].loglog(nn, 1 / (4 * nn ** 2), color=C["blue"], lw=1.2, ls="--"); ax[1].loglog(nn, 1 / (6 * nn ** 2), color=C["orange"], lw=1.2, ls="--")
    ax[1].set(xlabel="distance in grid units", ylabel="relative error of the lattice $1/r$", title="(b) error falls as $1/r^2$, with opposite sign\nalong axes ($+\\frac{1}{4r^2}$) and body diagonals ($-\\frac{1}{6r^2}$)"); ax[1].legend(frameon=False, fontsize=8); plt.tight_layout(); return fig, (ra, ea, rd, ed)
def fig_boundary(R, h=0.25):
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7)); Ls = (4, 8, 12); r = 8 * h; res = R["pair"]
    ax[0].semilogy(Ls, [abs(res[(L, 8)][0] - 1 / r) * r for L in Ls], "o-", color=C["red"], label="zero walls (potential = 0 at the box)"); ax[0].semilogy(Ls, [abs(res[(L, 8)][1] - 1 / r) * r for L in Ls], "s-", color=C["blue"], label="open (multipole) boundary values")
    ax[0].axhline(abs(R["Ginf"][8] - 1 / r) * r, color="gray", ls=":", label="lattice limit (infinite box)"); ax[0].set(xlabel="box half-size L [bohr]", ylabel="relative error of the pair potential at r = 2", title="(a) two charges: how the walls spoil $1/r$"); ax[0].legend(frameon=False, fontsize=7)
    sp = R["selfp"]; x = np.arange(len(sp)); ax[1].bar(x - 0.2, [abs(s[1]) for s in sp], 0.4, color=C["red"], label="zero walls"); ax[1].bar(x + 0.2, [max(abs(s[2]), 1e-4) for s in sp], 0.4, color=C["blue"], label="open boundary values"); ax[1].set_yscale("log")
    ax[1].set(xticks=x, xticklabels=[f"{s[0]:.2f}" for s in sp], xlabel="position of a single charge (fraction of the way to the wall)", ylabel="spurious self-potential [hartree]", title="(b) one charge acts on itself through the walls"); ax[1].axhline(0.5, color="k", ls="--", lw=0.8); ax[1].text(-0.4, 0.58, "a typical pair energy at r = 2", fontsize=7); ax[1].set_ylim(5e-5, 6); ax[1].legend(frameon=False, fontsize=7, loc="upper left"); plt.tight_layout(); return fig
def fig_harmonium(R):
    E_ex, E_hf = R["E"]; fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7)); x = np.linspace(-3.2, 3.2, 200); ax[0].plot(x, 0.5 * 0.25 * x ** 2, color="k"); ax[0].scatter([-1.1, 1.0], [0.0, 0.0], s=140, color=[C["blue"], C["orange"]], zorder=3, label="two electrons"); ax[0].annotate("", xy=(0.85, 0.02), xytext=(-0.95, 0.02), arrowprops=dict(arrowstyle="<->", color=C["red"])); ax[0].text(0, 0.07, r"repel: $1/r_{12}$", ha="center", color=C["red"], fontsize=9)
    ax[0].set(xlabel="position", ylabel="trap potential", title="(a) harmonium: two electrons in a harmonic trap", ylim=(-0.02, 1.3)); ax[0].legend(frameon=False, fontsize=8, loc="upper center")
    lv = [("no repulsion", 1.5, C["gray"]), ("exact (with repulsion)", E_ex, C["green"]), ("Hartree-Fock (mean field)", E_hf, C["orange"])]
    for i, (lab, e, col) in enumerate(lv): ax[1].hlines(e, i - 0.3, i + 0.3, color=col, lw=4); ax[1].text(i, e + 0.02, f"{e:.4f}", ha="center", fontsize=9)
    ax[1].set(xticks=range(3), xticklabels=[l[0] for l in lv], ylabel="ground-state energy [hartree]", ylim=(1.4, 2.15), title=f"(b) the energies: correlation energy = exact - HF = {E_ex - E_hf:+.4f}"); ax[1].tick_params(axis="x", labelsize=8); plt.tight_layout(); return fig
def fig_hf(R):
    rows = R["hf_scan"]; hs = np.array([r[0] for r in rows]); err = np.abs([r[1] for r in rows]); fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7))
    ax[0].loglog(hs, err, "o-", color=C["blue"], label="grid Hartree-Fock"); ax[0].loglog(hs, err[0] * (hs / hs[0]) ** 2, "k:", label=r"$h^2$ (second order)"); ax[0].set(xlabel="grid spacing h [bohr]", ylabel="|E(grid) - E(radial HF)| [hartree]", title="(a) error falls as $h^2$"); ax[0].legend(frameon=False, fontsize=8)
    box = R["box"]; Ls = sorted(box); ax[1].semilogy(Ls, [abs(box[L][0]) for L in Ls], "s-", color=C["blue"], label="open boundary values"); ax[1].semilogy(Ls, [abs(box[L][1]) for L in Ls], "o-", color=C["red"], label="zero walls"); ax[1].axhline(abs(R["E"][0] - R["E"][1]), color="gray", ls=":", label="|correlation energy|")
    ax[1].set(xlabel="box half-size L [bohr]", ylabel="|error| at h = 0.5", title="(b) the box: boundary conditions matter most"); ax[1].legend(frameon=False, fontsize=8); plt.tight_layout(); return fig
def fig_cancel(R):
    rows = R["hf_scan"]; sw = R["hf_swap"]; hs = [r[0] for r in rows]; fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7)); i = hs.index(0.5); x = np.arange(3)
    a = rows[i]; b = sw[i]; ax[0].bar(x - 0.2, [a[2], a[3], a[1]], 0.4, color=C["blue"], label="lattice self-value (as is)"); ax[0].bar(x + 0.2, [b[2], b[3], b[1]], 0.4, color=C["orange"], label="self-value swapped for cell average"); ax[0].axhline(0, color="k", lw=0.8)
    ax[0].set(xticks=x, xticklabels=["one-body part\n(kinetic + trap)", "Hartree part", "total"], ylabel="energy error at h = 0.5 [hartree]", title="(a) errors partly cancel; fixing one piece\nexposes the other"); ax[0].legend(frameon=False, fontsize=7)
    ax[1].loglog(hs, np.abs([r[1] for r in rows]), "o-", color=C["blue"], label="as is"); ax[1].loglog(hs, np.abs([r[1] for r in sw]), "s-", color=C["orange"], label="self-value swapped"); ax[1].loglog(hs, np.abs(rows[0][1]) * (np.array(hs) / hs[0]) ** 2, "k:", label=r"$h^2$"); ax[1].set(xlabel="grid spacing h [bohr]", ylabel="|energy error| [hartree]", title="(b) both converge as $h^2$"); ax[1].legend(frameon=False, fontsize=8); plt.tight_layout(); return fig
def fig_relative(R):
    rel = R["rel"]; hs = np.array([r[0] for r in rel]); fig, ax = plt.subplots(figsize=(5.8, 3.7)); ax.loglog(hs, np.abs([r[1] for r in rel]), "o-", color=C["blue"], label="interaction from the local constraint"); ax.loglog(hs, np.abs([r[2] for r in rel]), "s-", color=C["orange"], label="cell-averaged $1/r$ (the usual choice)"); ax.loglog(hs, np.abs(rel[0][1]) * (hs / hs[0]) ** 2, "k:", label=r"$h^2$")
    ax.set(xlabel="grid spacing h [bohr]", ylabel="|E - 2| [hartree]", title="exact two-electron energy on a grid"); ax.legend(frameon=False, fontsize=8); plt.tight_layout(); return fig
def fig_self(R):
    rows = R["self"]; hs = np.array([r[0] for r in rows]); fig, ax = plt.subplots(figsize=(6.2, 3.8))
    for k, (lab, col, mk) in enumerate((("pair sum, skip self", C["gray"], "o"), ("pair sum + cell-average self", C["green"], "^"), ("local solve (as is)", C["blue"], "s"), ("local solve, self swapped", C["orange"], "D"))): ax.loglog(hs, [r[2 + k] for r in rows], mk + "-", color=col, label=lab)
    ax.set(xlabel="grid spacing h [bohr]", ylabel="max error of the Hartree potential", title="the on-site (self) term dominates the error"); ax.legend(frameon=False, fontsize=8); plt.tight_layout(); return fig
def fig_cost(R):
    a = np.array(R["cost"]); ok = ~np.isnan(a[:, 2]); P = a[:, 1]; fig, ax = plt.subplots(figsize=(6.6, 4.0))
    ax.loglog(P[ok], a[ok, 2], "o-", color=C["red"], label="all-pairs sum"); ax.loglog(P, a[:, 3], "s-", color=C["blue"], label="fast solve (sine transform)"); ax.loglog(P, a[:, 4], "^-", color=C["orange"], label="conjugate gradients on the 7-point stencil")
    ex = np.polyfit(np.log(P[ok]), np.log(a[ok, 2]), 1); Pe = np.array([P[ok][0], P[-1]]); ax.loglog(Pe, np.exp(np.polyval(ex, np.log(Pe))), color=C["red"], ls=":", lw=1)
    ax.set(xlabel="number of grid points P", ylabel="time for one Hartree potential [s]", title=f"cost: all-pairs grows as P^{ex[0]:.2f}"); ax.legend(frameon=False, fontsize=8); plt.tight_layout(); return fig

In [ ]:
import time, os, sys
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
np.set_printoptions(precision=6, suppress=True, linewidth=120); R = {}

## 1. The cost of the pair sum
The potential at charge $i$ caused by all the others is $\sum_{j\ne i}q_j/|\mathbf r_i-\mathbf r_j|$. Doing it for every $i$ means $P(P-1)$ terms. We time this for increasing numbers of grid points $P$ (each grid point holds a little charge of a smooth blob).
**Stop and predict:** if $P$ is multiplied by $10$, by how much does the time grow?

In [ ]:
def pair_sum_time(N, h=0.5, w=0.5):
    c = (np.arange(N) - (N - 1) / 2) * h; X, Y, Z = np.meshgrid(c, c, c, indexing='ij'); P = np.stack([X.ravel(), Y.ravel(), Z.ravel()], 1); q = 2 * (w / np.pi) ** 1.5 * np.exp(-w * (X ** 2 + Y ** 2 + Z ** 2)).ravel() * h ** 3; t0 = time.time()
    for k in range(0, len(P), 500): d = np.sqrt(((P[k:k + 500, None, :] - P[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf; _ = (q[None, :] / d).sum(1)
    return time.time() - t0
ts = [(N ** 3, pair_sum_time(N)) for N in (9, 13, 17, 21)]; print("points   time [s]"); [print(f"{p:6d}   {t:7.3f}") for p, t in ts]
ex = np.polyfit(np.log([p for p, t in ts]), np.log([t for p, t in ts]), 1)[0]; print(f"time grows as P^{ex:.2f}: multiplying the number of points by 10 multiplies the time by about {10 ** ex:.0f}")
assert 1.7 < ex < 2.2

## 2. The local equation and its stencil
The potential $\phi$ of a charge density $\rho$ satisfies **Poisson's equation** $-\nabla^2\phi=4\pi\rho$ (atomic units). The **Laplacian** $\nabla^2$ measures how much $\phi$ at a point differs from the average of its surroundings. On a grid of spacing $h$ it becomes the **7-point stencil**
$$\frac{6\phi_{ijk}-\phi_{i+1,jk}-\phi_{i-1,jk}-\phi_{i,j+1,k}-\phi_{i,j-1,k}-\phi_{ij,k+1}-\phi_{ij,k-1}}{h^2}=4\pi\rho_{ijk}.$$
Each equation involves one point and its six neighbours, nothing else: the operator is **local** (a sparse matrix with seven entries per row). Yet solving *all* the equations together gives a potential that reaches the whole grid, just as a heat source warms a whole room through neighbour-to-neighbour conduction.
**Stop and predict:** if one unit charge sits in the middle of a grid, how will the potential at 2, 4 and 8 cells away compare with $1/r$?

In [ ]:
fig_stencil(); plt.show()
N, h = 41, 1.0; A = stencil_matrix(N, h); c = (np.arange(N) - N // 2) * h; rho = np.zeros((N, N, N)); rho[N // 2, N // 2, N // 2] = 1 / h ** 3; rhs = 4 * np.pi * rho
W = (N // 2 + 1) * h; U, V = np.meshgrid(c, c, indexing="ij"); wall = lambda a, b, d: 1 / np.sqrt(a ** 2 + b ** 2 + d ** 2)
for s, sl in ((-1, 0), (1, -1)): rhs[sl, :, :] += wall(s * W, U, V) / h ** 2; rhs[:, sl, :] += wall(U, s * W, V) / h ** 2; rhs[:, :, sl] += wall(U, V, s * W) / h ** 2
phi, its = cg_solve(A, rhs.ravel(), 1e-10); phi = phi.reshape(N, N, N); print(f"matrix: {A.shape[0]} unknowns, {A.nnz / A.shape[0]:.1f} nonzeros per row; conjugate gradients converged in {its} iterations")
for r in (2, 4, 8): p = phi[N // 2 + r, N // 2, N // 2]; print(f"distance {r} cells: potential {p:.5f}  exact 1/r = {1 / (r * h):.5f}  relative error {abs(p * r * h - 1):.4f}")
print(f"in the charge's own cell the potential is {phi[N // 2, N // 2, N // 2]:.4f} = 4 pi G(0)/h (next section)")
assert abs(phi[N // 2 + 8, N // 2, N // 2] * 8 - 1) < 0.01 and 6.5 < A.nnz / A.shape[0] <= 7.0

## 3. The lattice Green function: what the grid does to $1/r$
For a unit charge in one cell, the stencil equations have the solution $\phi(\mathbf n)=\frac{4\pi}{h}G(\mathbf n)$ at the cell $\mathbf n=(n_1,n_2,n_3)$ (in grid units), where $G$ is the **lattice Green function** of the 7-point operator, $G(\mathbf n)=\int_0^\infty e^{-6t}I_{n_1}(2t)I_{n_2}(2t)I_{n_3}(2t)\,dt$ ($I_n$: modified Bessel functions). Far away $G\to1/(4\pi|\mathbf n|)$, so the interaction tends to $1/r$; at short range it differs, and its value at the origin, $4\pi G(0)/h=3.176/h$, is the **self-value** (the charge acting on its own cell, finite where $1/r$ is infinite). A known expansion says the relative error is $\frac{1}{8r^2}\bigl(5\frac{n_1^4+n_2^4+n_3^4}{r^4}-3\bigr)$: it falls as $1/r^2$ and depends on the direction, because a cubic grid is not rotationally symmetric.
**Stop and predict:** is the lattice potential slightly larger or smaller than $1/r$ along a grid axis? And along the body diagonal $(1,1,1)$?

In [ ]:
g = lattice_G(np.array([[0, 0, 0], [1, 0, 0], [1, 1, 1]])); print(f"G(0,0,0) = {g[0]:.7f}  (literature 0.2527310)"); print(f"defining identity 6 G(0) - 6 G(1,0,0) = {6 * g[0] - 6 * g[1]:.7f}  (exactly 1)"); print(f"self-value 4 pi G(0) = {4 * np.pi * g[0]:.4f} (in units of 1/h)")
rows = []
for n in (4, 8, 16, 32):
    ga = lattice_G(np.array([[n, 0, 0]]))[0]; gd = lattice_G(np.array([[n, n, n]]))[0]; ra, rd = n, n * np.sqrt(3); rows.append((n, 4 * np.pi * ra * ga - 1, (5 - 3) / (8 * ra ** 2), 4 * np.pi * rd * gd - 1, (5 / 3 - 3) / (8 * rd ** 2)))
import pandas as pd; print(pd.DataFrame(rows, columns=["n", "axis: error", "axis: expansion", "diagonal: error", "diagonal: expansion"]).round(5).to_string(index=False))
fig, _ = fig_lattice(lattice_G); plt.show(); R['G'] = g
print("along an axis the lattice potential is a little too large (+1/(4 r^2)), along the body diagonal a little too small (-1/(6 r^2)): the grid interaction is slightly anisotropic, and the expansion reproduces the numbers")
assert abs(g[0] - 0.252731) < 2e-6 and abs(6 * g[0] - 6 * g[1] - 1) < 2e-6 and rows[2][1] > 0 > rows[2][3] and abs(rows[2][1] - rows[2][2]) < 5e-5 and abs(rows[2][3] - rows[2][4]) < 5e-5

## 4. Boundaries: zero walls versus open boundary values
A real computation lives in a finite box, and the equations on the outermost points need *values* just outside. The simplest choice, **zero walls** ($\phi=0$ on the box surface), is wrong for charges: it behaves as if the box were a metal shell, which creates image charges of opposite sign that screen the interaction. The better choice for a localized charge is **open boundary values**: put on the walls the potential of the total charge and its dipole, $Q/r+\mathbf p\cdot\mathbf r/r^3$ (the first two terms of the **multipole expansion**).
**Stop and predict:** two unit charges $r=2$ bohr apart in a box. Will the error of their interaction shrink or grow as the box gets larger, for zero walls and for open boundary values?

In [ ]:
h = 0.25
def grid(L): N = int(round(2 * L / h)) + 1; return N, (np.arange(N) - (N - 1) / 2) * h
def point(N, ix, iy, iz, q=1.0): rho = np.zeros((N, N, N)); rho[ix, iy, iz] = q / h ** 3; return rho
seps = (2, 4, 8, 16); res = {}
for L in (4, 8, 12):
    N, c = grid(L); m = N // 2
    for sep in seps:
        i1, i2 = m - sep // 2, m + sep // 2; rho = point(N, i1, m, m); res[(L, sep)] = (poisson_dirichlet(rho, h)[i2, m, m], poisson_dirichlet(rho, h, multipole_bc(1.0, np.array([c[i1], 0, 0])))[i2, m, m])
R['pair'] = res; R['Ginf'] = {s: 4 * np.pi * lattice_G(np.array([[s, 0, 0]]))[0] / h for s in seps}
print("relative error of the pair potential against 1/r        r   lattice only | L=8: zero walls, open | L=12: zero walls, open")
for s in seps: r = s * h; print(f"                                                      {r:5.2f}   {abs(R['Ginf'][s] - 1 / r) * r:9.1e}   |  {abs(res[(8, s)][0] - 1 / r) * r:9.1e} {abs(res[(8, s)][1] - 1 / r) * r:9.1e}  |  {abs(res[(12, s)][0] - 1 / r) * r:9.1e} {abs(res[(12, s)][1] - 1 / r) * r:9.1e}")
Ginf0 = 4 * np.pi * G0 / h; N, c = grid(8); m = N // 2; selfp = []
for frac in (0.0, 0.5, 0.8, 0.95):
    ix = m + int(round(frac * (N // 2 - 1))); rho = point(N, ix, m, m); selfp.append((frac, poisson_dirichlet(rho, h)[ix, m, m] - Ginf0, poisson_dirichlet(rho, h, multipole_bc(1.0, np.array([c[ix], 0, 0])))[ix, m, m] - Ginf0))
R['selfp'] = selfp; fig_boundary(R); plt.show()
print("zero walls: the error decays only like 1/L (images screen the charge); open boundary values: only the lattice error remains. A single charge also feels a spurious potential from the walls (panel b), comparable to a pair energy when it sits near them: keep the charge well inside the box")
assert abs(res[(8, 8)][0] - 1 / 2) * 2 > 0.1 and abs(res[(8, 8)][1] - 1 / 2) * 2 < 0.01

## 5. Harmonium: the exact and the mean-field energies
**Harmonium** is the simplest interacting system with a known exact answer: two electrons in a harmonic trap $\tfrac12\omega^2r^2$ that repel each other. We use $\omega=\tfrac12$, for which the exact ground-state energy is exactly $2$ hartree. Its motion separates into the centre of mass (a free harmonic oscillator) and the **relative coordinate** $\mathbf r=\mathbf r_1-\mathbf r_2$ ($H_{\rm rel}=-\nabla^2+\tfrac14\omega^2r^2+1/r$); $E=\tfrac32\omega+\varepsilon_{\min}$.
A **mean-field** (Hartree–Fock, HF) calculation instead lets each electron feel the *average* potential of the other (the **Hartree potential**, $v_H(\mathbf r)=\int n(\mathbf r')/|\mathbf r-\mathbf r'|\,d^3r'$) and solves for the orbital self-consistently. The energy it misses is the **correlation energy**, exact minus HF: here $-0.0384$ hartree. Because the constraint can only change *how the Hartree potential is computed*, it cannot recover correlation; what we test is whether it reproduces the HF energy.
**Stop and predict:** without the repulsion the two electrons have energy $2\cdot\tfrac32\omega=1.5$. Is the exact energy with repulsion above or below the HF energy?

In [ ]:
E_exact, E_HF, T_hf, J_hf = None, None, None, None
Ex, E_hf, T_hf, J_hf = radial_reference(parts=True); E_exact = Ex; R['E'] = (Ex, E_hf); R['HFparts'] = (T_hf, J_hf)
print(f"non-interacting energy of two electrons   1.5000\nexact two-electron energy                 {Ex:.7f}   (analytic: 2)\nHartree-Fock energy                       {E_hf:.7f}   (literature: 2.0384)\ncorrelation energy                        {Ex - E_hf:+.6f}")
fig_harmonium(R); plt.show()
print(f"the HF energy splits into a one-body part 2(kinetic + trap) = {T_hf:.6f} and the Hartree repulsion J = {J_hf:.6f}")
print("the exact energy lies BELOW HF (HF is a variational upper bound for a good trial state) by the correlation energy")
assert abs(Ex - 2) < 1e-6 and abs(E_hf - 2.0384388) < 1e-5

## 6. Mean-field harmonium on a 3D grid
Now the orbital lives on an $N^3$ grid with the 7-point kinetic energy, and at every self-consistency step the Hartree potential comes from the **local constraint** (the Poisson solve, with open boundary values). We scan the grid spacing $h$ and the box size $L$. A method is of **order $p$** if its error falls like $h^p$; on a log-log plot that is a straight line of slope $p$. **Richardson extrapolation** uses two spacings to cancel the $h^2$ term.
**Stop and predict:** what slope will the error show, and what error remains after Richardson extrapolation of the two finest grids?

In [ ]:
t0 = time.time(); HV = (0.6, 0.5, 0.4, 0.3, 0.25); rows = []
for h_ in HV: E, it, N, T, J = grid_hf(h_, 6, 'multipole', parts=True); rows.append((h_, E - E_hf, T - T_hf, J - J_hf))
R['hf_scan'] = rows; err = [r[1] for r in rows]; sl = [np.log(abs(err[i + 1] / err[i])) / np.log(HV[i + 1] / HV[i]) for i in range(len(HV) - 1)]
print("  h     N^3    total error     one-body part (kinetic + trap)   Hartree part"); [print(f"  {r[0]:4.2f}  {int(round(12 / r[0])) + 1:3d}^3   {r[1]:+.3e}        {r[2]:+.3e}                  {r[3]:+.3e}") for r in rows]
print("log-log slope of the total error:", np.round(sl, 2)); (h1, a1), (h2, a2) = (HV[-2], err[-2]), (HV[-1], err[-1]); cc = (a2 - a1) / (h2 ** 2 - h1 ** 2); print(f"Richardson (h^2) extrapolation of the two finest grids: error {a2 - cc * h2 ** 2:+.1e}  [{time.time() - t0:.0f} s]")
box = {}
for L in (3, 4, 5, 6, 8, 10): box[L] = (grid_hf(0.5, L, 'multipole')[0] - E_hf, grid_hf(0.5, L, 'zero')[0] - E_hf)
R['box'] = box; fig_hf(R); plt.show()
print("box scan at h = 0.5: open boundary values converge once the density fits (L >= 5); zero walls are wrong by several times the correlation energy and recover only like 1/L")
assert all(1.9 < s < 2.1 for s in sl) and abs(a2 - cc * h2 ** 2) < 1e-5 and abs(box[10][1]) > abs(Ex - E_hf) and abs(box[6][0]) < 0.01

**An error budget.** The total grid error splits into the error of the one-body part (finite-difference kinetic energy and trap) and the error of the Hartree part. The two have *opposite signs*: the one-body part is too low and the Hartree part too high, so part of each cancels. Replacing the on-site value of the lattice interaction ($3.176/h$) by the cell average of $1/r$ ($2.380/h$) improves the Hartree potential (next sections) but removes some of this helpful cancellation.
**Stop and predict:** if the Hartree part becomes smaller (less positive), does the total error get better or worse?

In [ ]:
t0 = time.time(); sw = []
for h_ in HV: E, it, N, T, J = grid_hf(h_, 6, 'multipole', selfcorr=True, parts=True); sw.append((h_, E - E_hf, T - T_hf, J - J_hf))
R['hf_swap'] = sw; fig_cancel(R); plt.show(); i = HV.index(0.5)
print("at h = 0.5:   one-body   Hartree    total"); print(f"as is        {rows[i][2]:+.2e}  {rows[i][3]:+.2e}  {rows[i][1]:+.2e}"); print(f"self swapped {sw[i][2]:+.2e}  {sw[i][3]:+.2e}  {sw[i][1]:+.2e}   [{time.time() - t0:.0f} s]")
print(f"the Hartree error offsets {-rows[i][3] / rows[i][2]:.0%} of the one-body error as is, and only {-sw[i][3] / sw[i][2]:.0%} after the swap: the total gets worse; compare total energies, and extrapolate in h^2")
assert all(r[2] < 0 < r[3] for r in rows) and abs(sw[i][1]) > abs(rows[i][1])

## 7. The exact two-electron problem with the grid interaction
Because harmonium separates, the exact two-electron problem is a *single* 3D problem in the relative coordinate $\mathbf r$, with the interaction $V(\mathbf r)$ on the grid. We compare two choices of $V$ against the exact $E=2$: the interaction **generated by the constraint** ($V=4\pi G(\mathbf n)/h$, translation-invariant) and the usual **cell-averaged** $1/r$ (point value $1/r$, with $2.380/h$ at the origin). This tests the interaction the constraint produces, including the electron–electron cusp ($1/r$ is singular at $r=0$), though not its cost advantage.
**Stop and predict:** will the cusp spoil the second-order convergence?

In [ ]:
t0 = time.time(); rel = []
for h_ in (0.6, 0.5, 0.4, 0.3): Ec, N = relative_problem(h_, potential='lattice'); Ecell, _ = relative_problem(h_, potential='cell'); rel.append((h_, Ec - 2, Ecell - 2))
R['rel'] = rel; hs = np.array([r[0] for r in rel]); print("   h     N^3    constraint-generated: error    cell-averaged: error    ratio"); [print(f"  {r[0]:4.2f}  {int(round(20 / r[0])) + 1:3d}^3      {r[1]:+.3e}                 {r[2]:+.3e}          {r[2] / r[1]:.2f}") for r in rel]
for k, name in ((1, "constraint"), (2, "cell-averaged")): print(f"  {name}: slopes", np.round(np.diff(np.log(np.abs([r[k] for r in rel]))) / np.diff(np.log(hs)), 2))
fig_relative(R); plt.show(); print(f"[{time.time() - t0:.0f} s]  both are second order: the cusp does not spoil convergence; the constraint-generated interaction has about half the error (a plausible, untested reason: it is the Green function of the same lattice operator that defines the kinetic energy)")
assert all(1.9 < s < 2.1 for s in np.diff(np.log(np.abs([r[1] for r in rel]))) / np.diff(np.log(hs))) and all(1.7 < r[2] / r[1] < 2.2 for r in rel)

## 8. The self-term
The solve includes each cell's *own* contribution, $4\pi G(0)/h=3.176/h$ times its charge, where a pair sum that skips $j=i$ includes none, and a cell-averaged treatment would use $2.380/h$. For a smooth density this on-site term dominates the error. We compare, against the analytic Hartree potential of a Gaussian density, a pair sum without self, a pair sum with the cell-average self term, the stencil solve, and the stencil with its self value swapped for the cell average.
**Stop and predict:** which of the four is most accurate for the potential?

In [ ]:
from scipy.special import erf
t0 = time.time(); wg = 0.5; Gl0 = 4 * np.pi * G0; analytic = lambda Rr: np.where(Rr > 1e-12, 2 * erf(np.sqrt(wg) * Rr) / np.where(Rr > 1e-12, Rr, 1), 4 * np.sqrt(wg / np.pi)); srows = []
for h_ in (1.0, 0.75, 0.5):
    L = 6.0; N = int(round(2 * L / h_)) + 1; c = (np.arange(N) - (N - 1) / 2) * h_; X, Y, Z = np.meshgrid(c, c, c, indexing='ij'); Rr = np.sqrt(X ** 2 + Y ** 2 + Z ** 2); rho = 2 * (wg / np.pi) ** 1.5 * np.exp(-wg * Rr ** 2); P = np.stack([X.ravel(), Y.ravel(), Z.ravel()], 1); q = rho.ravel() * h_ ** 3; phid = np.empty(len(P))
    for k in range(0, len(P), 400): d = np.sqrt(((P[k:k + 400, None, :] - P[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf; phid[k:k + 400] = (q[None, :] / d).sum(1)
    an = analytic(Rr.ravel()); ps = poisson_dirichlet(rho, h_, multipole_bc(rho.sum() * h_ ** 3, [0, 0, 0])).ravel(); ps2 = ps - Gl0 * h_ ** 2 * rho.ravel() + CELL0 * h_ ** 2 * rho.ravel()
    srows.append((h_, N, np.abs(phid - an).max(), np.abs(phid + CELL0 * h_ ** 2 * rho.ravel() - an).max(), np.abs(ps - an).max(), np.abs(ps2 - an).max()))
R['self'] = srows; print("     h    N   pair sum (skip self) | pair sum + cell-avg self | stencil solve | stencil, self swapped"); [print(f"  {r[0]:5.3f} {r[1]:3d}      {r[2]:.2e}              {r[3]:.2e}            {r[4]:.2e}      {r[5]:.2e}") for r in srows]
fig_self(R); plt.show(); print(f"[{time.time() - t0:.0f} s]  the on-site term dominates; the stencil with a swapped self value is the most accurate Hartree potential, yet (Section 6) it gives a worse energy: do not tune one ingredient in isolation")
assert all(r[5] < r[4] < r[2] for r in srows)

## 9. The cost of the local solve
The pair sum costs $\propto P^2$. The local equation can be solved in two ways, both using only the local operator: by an **iterative solver** (conjugate gradients, CG, which repeatedly applies the sparse stencil matrix) or by a **fast sine transform**, which diagonalizes the same operator and costs $\propto P\log P$. We check that both give the same potential and time all three.
**Stop and predict:** how many CG iterations will be needed as the grid gets bigger, and what does that imply for the cost?

In [ ]:
t00 = time.time(); hc = 0.5; wc = 0.5; crow = []
for N in (9, 13, 17, 21, 25, 33, 41, 49, 65):
    c = (np.arange(N) - (N - 1) / 2) * hc; X, Y, Z = np.meshgrid(c, c, c, indexing='ij'); rho = 2 * (wc / np.pi) ** 1.5 * np.exp(-wc * (X ** 2 + Y ** 2 + Z ** 2)); bc = multipole_bc(rho.sum() * hc ** 3, [0, 0, 0]); td = pair_sum_time(N) if N <= 25 else np.nan
    t0 = time.time(); phi_f = poisson_dirichlet(rho, hc, bc); tf = time.time() - t0; A = stencil_matrix(N, hc); rhs = 4 * np.pi * rho.copy(); xb = (N + 1) / 2 * hc; A2, B2 = np.meshgrid(c, c, indexing='ij'); Wp = np.full_like(A2, xb)
    for sgn, sl_ in ((-1, 0), (1, -1)): rhs[sl_, :, :] += bc(sgn * Wp, A2, B2) / hc ** 2; rhs[:, sl_, :] += bc(A2, sgn * Wp, B2) / hc ** 2; rhs[:, :, sl_] += bc(A2, B2, sgn * Wp) / hc ** 2
    t0 = time.time(); phi_c, its = cg_solve(A, rhs.ravel(), 1e-12); tc = time.time() - t0; crow.append((N, N ** 3, td, tf, tc, its, np.abs(phi_f.ravel() - phi_c).max()))
R['cost'] = crow; a = np.array(crow); ok = ~np.isnan(a[:, 2]); print("    N   points   pair sum [s]   fast solve [s]   CG [s] (iterations)   |fast - CG|"); [print(f"  {int(r[0]):3d}  {int(r[1]):7d}  {r[2]:9.3f}    {r[3]:10.4f}    {r[4]:7.3f} ({int(r[5]):4d})      {r[6]:.1e}") for r in crow]
ex_pair = np.polyfit(np.log(a[ok, 1]), np.log(a[ok, 2]), 1)[0]; ex_cg = np.polyfit(np.log(a[3:, 1]), np.log(a[3:, 4]), 1)[0]; ex_f = np.polyfit(np.log(a[3:, 1]), np.log(a[3:, 3]), 1)[0]
print(f"empirical exponents of the cost in the number of points P: pair sum {ex_pair:.2f}, CG {ex_cg:.2f}, sine transform {ex_f:.2f}; CG iterations / N = {np.round(a[:, 5] / a[:, 0], 2)} (about 3.2 N = 3.2 P^(1/3))")
fig_cost(R); plt.show(); t_ex = a[ok, 2][-1] * (a[-1, 1] / a[ok, 1][-1]) ** ex_pair; print(f"extrapolated: the pair sum for the largest grid (P = {int(a[-1, 1])}) would take about {t_ex / 60:.0f} minutes against {a[-1, 3]:.3f} s for the fast solve [{time.time() - t00:.0f} s]")
assert 1.7 < ex_pair < 2.2 and 1.1 < ex_cg < 1.6 and a[:, 6].max() < 1e-10 and 2.5 < np.mean(a[:, 5] / a[:, 0]) < 4

## 10. Your turn
**Exercise 1.** In Section 2 move the box walls from $N=41$ to $N=21$ at fixed $h$. What happens to the potential at $r=8$, and why?
**Exercise 2.** Use `lattice_G` to compute $4\pi rG$ along the direction $(1,1,0)$ and compare with the expansion $\frac1{8r^2}(5\sum n_i^4/r^4-3)$.
**Exercise 3.** In Section 4 add a *dipole-free* check: put the two charges off-centre and see which boundary values are needed.
**Exercise 4.** Repeat Section 6 with $\omega=1$ (the density is narrower). How do the error and the box requirement change?
**Exercise 5.** Replace the Gaussian in Section 8 by a density with a sharper peak and see how the self-term error scales with $h$.

## 11. Self-check

In [ ]:
assert abs(g[0] - 0.252731) < 2e-6 and abs(Ex - 2) < 1e-6 and all(1.9 < s < 2.1 for s in sl) and all(r[2] < 0 < r[3] for r in rows) and a[:, 6].max() < 1e-10
print("All checks passed: you reproduced the numbers in the post.")

# Part II (optional): independent checks with SciPy only
> **Read this first.** Part II checks pieces of Part I by independent routes. **(1)** The lattice Green function $G(0,0,0)$ by direct numerical integration (`scipy.integrate.quad`) of $\int_0^\infty e^{-6t}I_0(2t)^3dt$, against the value used above and the literature value $0.2527310$. **(2)** The Poisson solve: the residual of the 7-point equations, evaluated with the explicit sparse matrix, for the potential produced by the fast solver. **(3)** Superposition: a linear equation must give the potential of two charge distributions as the sum of their separate potentials (the solver and the open boundary values are both linear). **(4)** The literature values $E_{\rm exact}=2$ and $E_{\rm HF}=2.0384$ (typed in) against Section 5. Each stage is guarded; the last cell lists the checks that passed.

In [ ]:
checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
try:
    from scipy.integrate import quad; from scipy.special import ive
    T = 2e4; val, err = quad(lambda t: ive(0, 2 * t) ** 3, 0, 1, limit=200); v2, _ = quad(lambda t: ive(0, 2 * t) ** 3, 1, T, limit=400); G_quad = val + v2 + 2 / (4 * np.pi) ** 1.5 / np.sqrt(T)
    print(f"G(0,0,0) by scipy.integrate.quad (with the analytic tail): {G_quad:.6f}; used above: {R['G'][0]:.6f}; literature 0.252731"); check("G(0,0,0) by direct quadrature agrees with the value used above (to 1e-5)", abs(G_quad - R["G"][0]) < 1e-5)
except Exception as e_: print("quadrature stage could not be completed:", type(e_).__name__, e_)
try:
    N, h = 25, 0.5; c = (np.arange(N) - (N - 1) / 2) * h; X, Y, Z = np.meshgrid(c, c, c, indexing='ij'); rho = 2 * (0.5 / np.pi) ** 1.5 * np.exp(-0.5 * (X ** 2 + Y ** 2 + Z ** 2)); bc = multipole_bc(rho.sum() * h ** 3, [0, 0, 0]); phi = poisson_dirichlet(rho, h, bc)
    pad = np.pad(phi, 1); xb = (N + 1) / 2 * h; A2, B2 = np.meshgrid(c, c, indexing='ij'); Wp = np.full_like(A2, xb)
    for sgn, ix in ((-1, 0), (1, -1)): pad[ix, 1:-1, 1:-1] = bc(sgn * Wp, A2, B2); pad[1:-1, ix, 1:-1] = bc(A2, sgn * Wp, B2); pad[1:-1, 1:-1, ix] = bc(A2, B2, sgn * Wp)
    lap = (6 * pad[1:-1, 1:-1, 1:-1] - pad[2:, 1:-1, 1:-1] - pad[:-2, 1:-1, 1:-1] - pad[1:-1, 2:, 1:-1] - pad[1:-1, :-2, 1:-1] - pad[1:-1, 1:-1, 2:] - pad[1:-1, 1:-1, :-2]) / h ** 2
    print(f"largest residual of the 7-point equations, explicit stencil applied to the fast-solver potential: {np.abs(lap - 4 * np.pi * rho).max():.1e}"); check("the fast solver satisfies the explicit 7-point equations (residual below 1e-10)", np.abs(lap - 4 * np.pi * rho).max() < 1e-10)
    rho2 = np.zeros_like(rho); rho2[18, 9, 14] = 1 / h ** 3; Q1, Q2 = rho.sum() * h ** 3, rho2.sum() * h ** 3; p1 = [(rho * A_).sum() * h ** 3 for A_ in (X, Y, Z)]; p2 = [(rho2 * A_).sum() * h ** 3 for A_ in (X, Y, Z)]
    phi_sum = poisson_dirichlet(rho, h, multipole_bc(Q1, p1)) + poisson_dirichlet(rho2, h, multipole_bc(Q2, p2)); phi_both = poisson_dirichlet(rho + rho2, h, multipole_bc(Q1 + Q2, np.add(p1, p2)))
    print(f"largest difference between the potential of the two charge distributions together and the sum of their separate potentials: {np.abs(phi_both - phi_sum).max():.1e}"); check("superposition holds (difference below 1e-12)", np.abs(phi_both - phi_sum).max() < 1e-12)
except Exception as e_: print("Poisson stage could not be completed:", type(e_).__name__, e_)
check("Section 5 reproduces the literature energies (exact 2, Hartree-Fock 2.0384)", abs(R["E"][0] - 2) < 1e-6 and abs(R["E"][1] - 2.0384) < 5e-5)
if checks:
    ok_ = [l for l, p_ in checks if p_]; bad_ = [l for l, p_ in checks if not p_]; print(f"\nPart II: {len(ok_)} of {len(checks)} checks passed")
    for l in bad_: print("  not passed:", l)